# Native LTX-2.5 A2V lip-sync LoRA training on one Colab GPU

This notebook runs the repository's native ltx-trainer path for the specific contract:

**one supplied first-frame image + one clean, time-aligned speech recording → generated talking-head video with generated mouth motion.**

It is deliberately pinned to a known repository revision and keeps the raw dataset immutable. The notebook:

1. prepares a reproducible Colab runtime;
2. clones the exact LTX-2.5 trainer revision;
3. downloads the matching split model pack after Hugging Face authentication;
4. creates a deterministic training/holdout split;
5. extracts the held-out first frame and audio for first-frame QA;
6. precomputes video latents, audio latents, and caption embeddings with the official dataset processor;
7. materializes and validates the A2V LoRA configuration;
8. runs a short smoke test before the full run;
9. validates the produced LoRA checkpoint; and
10. optionally runs the native two-stage A2V pipeline with the trained LoRA.

The default is a 20-step smoke test. Change SMOKE_TEST to False only after the smoke checkpoint and validation sample look correct.

This notebook does not contain API keys, model weights, or generated outputs. Accept the LTX-2.5 model terms on Hugging Face before running it.

## Handoff: experiment contract and boundaries

This notebook is an implementation experiment for the exact inference contract: a user supplies a still image and a speech recording, and LTX-2.5 generates the video while the supplied speech remains the audio source. The trainable modality is video. The audio branch is present as clean, frozen conditioning, so the loss teaches the video branch to use speech features; this is the causal direction required for talking-head lip synchronization.

Each raw training row is one paired talking clip. The processor reads the video, extracts its original audio from the same container, trims or pads that audio to the selected VAE-aligned video duration, and writes matching video latents, audio latents, and text embeddings. The first latent video frame is used as the first-frame condition with probability 1.0. At inference, the equivalent condition is a supplied image encoded as the first frame.

This is native LTX-2.5 LoRA training. It is not AI Toolkit training, not a separate lip-sync decoder, and not a voice or identity LoRA. The adapter is attached to the LTX transformer attention paths, including audio-to-video attention, while the base model, VAEs, and text encoder remain frozen. The output adapter is saved in the ComfyUI-compatible diffusion_model namespace.

The current repository dataset is a small engineering dataset. The one-clip holdout is useful for checking that the pipeline, conditioning direction, timing, and checkpoint format work. It is not a true unseen-speaker generalization benchmark. A production-quality model would need a speaker-disjoint validation set, more speakers, more phonetic coverage, and consistent source quality. Do not read a successful smoke sample as evidence of 99/100 generalization.

The required model set is the matching LTX-2.5 dev transformer, LTX-specific Gemma 4 text encoder, video VAE, and audio VAE. The optional native two-stage inference cell additionally needs the official distilled LoRA and spatial upsampler. Those official inference files are separate from the LoRA trained here.

## Handoff: operator runbook

### Before starting

1. Use a Linux Colab GPU runtime and confirm sufficient disk. High VRAM helps with the 22B dev transformer, but VRAM does not provide extra disk space.
2. Accept the LTX-2.5 gated model terms on Hugging Face. Use a read token through the hidden prompt or a Colab secret; never paste a token into a notebook cell.
3. Review the editable controls in the first code cell. The default is a 20-step smoke run at 1280x704x153 and 25 fps.
4. Keep the raw repository dataset immutable. The notebook stages symlinks and writes all run-specific files under /content/ltx2.5_runs.

### Execute in order

Run the cells top to bottom. The preflight, clone, installation, and model cells establish the environment. The manifest cell creates a deterministic train/holdout split. The stream audit must report exactly one video and one audio stream per row. The first-frame QA cell then displays frame 0, nearby frames at 0.2s, 0.4s, and 1.0s, plus the held-out audio. Stop if the intended speaker is not clear, the mouth is already clipped, or the speech starts mid-word.

Preprocessing is a required boundary, not an optional optimization. Do not manually create a second audio manifest or independently normalize the audio: the official processor extracts the paired audio and uses the matching audio VAE. The cache audit must show the same number of files in latents, audio_latents, and conditions.

The configuration cell copies the reviewed trainer profile and fills only runtime paths, validation media, step counts, and output paths. It refuses a profile that does not generate video, freeze audio, apply first-frame conditioning at probability 1.0, or target audio-to-video attention. The training cell invokes scripts/train.py directly on one visible GPU. It does not use accelerate launch.

### Promote smoke to full training

First complete the smoke run and inspect its checkpoint and validation samples. To continue that same run into the full experiment, keep RUN_NAME unchanged, set SMOKE_TEST=False and START_FRESH=False, rerun the configuration cell, then rerun training. The notebook will load the latest smoke LoRA and its minimal training state, then continue to the full step count. To start an independent full run, choose a new RUN_NAME and set START_FRESH=True; that intentionally creates a new run-specific cache and output directory.

Never set START_FRESH=True against a non-empty output directory. Never set START_FRESH=False unless a checkpoint exists. If a run is interrupted, resume the same run rather than silently starting at step zero.

In [ ]:
from __future__ import annotations

import datetime as dt
import hashlib
import json
import os
import random
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path

# --------- User-editable experiment controls ---------

REPO_URL = "https://github.com/Yuvrajxms09/LTX-2.git"
# Pinned to the verified trainer + dataset revision. Update this deliberately
# when a newer fork revision has been reviewed.
REPO_REF = "728b3da48ce6ea5e2b6509e23e4e65c951d5b8f1"

WORK_ROOT = Path("/content/ltx2.5")
RUNS_ROOT = Path("/content/ltx2.5_runs")
MODEL_ROOT = Path("/content/models/ltx-2.5")
HF_CACHE_ROOT = Path("/content/hf_cache")

RUN_NAME = "a2v_lipsync_lora_smoke_35"
SMOKE_TEST = True
SMOKE_STEPS = 20
TRAIN_STEPS = 250
LORA_RANK = 16
LORA_ALPHA = 16
LEARNING_RATE = 5.0e-5
START_FRESH = True
PRECOMPUTE_OVERWRITE = False
RUN_TRAINER_VALIDATION = False
RUN_PIPELINE_SMOKE_TEST = False
DOWNLOAD_LORA = False

# Dataset split. A held-out source clip is used only for validation/inference.
HOLDOUT_SEED = 17
HOLDOUT_COUNT = 1
HOLDOUT_VIDEO_NAME = None  # Set to a manifest filename to choose a specific holdout.

# Native LTX-2.5 bucket. 153 = 8 * 19 + 1, which is VAE-aligned.
VIDEO_BUCKET = "1280x704x153"
WIDTH, HEIGHT, NUM_FRAMES = 1280, 704, 153
FRAME_RATE = 25.0
PREPROCESS_BATCH_SIZE = 1
NUM_DATALOADER_WORKERS = 2
MIN_FREE_DISK_GIB = 80

# --------- Derived paths ---------

REPO_DIR = WORK_ROOT / "LTX-2"
RUN_ROOT = RUNS_ROOT / RUN_NAME
TRAINER_DIR = REPO_DIR / "packages" / "ltx-trainer"
CONFIG_TEMPLATE = TRAINER_DIR / "configs" / "a2v_lipsync_lora.yaml"

SOURCE_DATASET_DIR = REPO_DIR / "datasets" / "training_clips_612"
SOURCE_MANIFEST = SOURCE_DATASET_DIR / "dataset_manifest.jsonl"

RUNTIME_DATASET_DIR = RUN_ROOT / "dataset"
RUNTIME_MEDIA_DIR = RUNTIME_DATASET_DIR / "media"
TRAIN_MANIFEST = RUNTIME_DATASET_DIR / "train_manifest.jsonl"
HOLDOUT_MANIFEST = RUNTIME_DATASET_DIR / "holdout_manifest.jsonl"
SPLIT_SUMMARY = RUNTIME_DATASET_DIR / "split_summary.json"

PRECOMPUTED_DIR = RUN_ROOT / "precomputed"
PREPROCESS_SPEC = PRECOMPUTED_DIR / "preprocess_spec.json"

VALIDATION_DIR = RUN_ROOT / "validation"
VAL_IMAGE = VALIDATION_DIR / "holdout_first_frame.png"
VAL_AUDIO = VALIDATION_DIR / "holdout_audio.wav"

OUTPUT_DIR = RUN_ROOT / "outputs"
RUN_CONFIG = RUN_ROOT / "run_config.yaml"
RUN_METADATA = RUN_ROOT / "run_metadata.json"
PIPELINE_OUTPUT = RUN_ROOT / "native_a2v_smoke.mp4"

TRANSFORMER_PATH = MODEL_ROOT / "diffusion_models" / "ltx-2.5-22b-dev-transformer-bf16.safetensors"
TEXT_ENCODER_PATH = MODEL_ROOT / "text_encoders" / "gemma4-12b-with-proj-ltx-2.5-bf16.safetensors"
VIDEO_VAE_PATH = MODEL_ROOT / "vae" / "ltx-2.5-video-vae-bf16.safetensors"
AUDIO_VAE_PATH = MODEL_ROOT / "vae" / "ltx-2.5-audio-vae-bf16.safetensors"
SPATIAL_UPSAMPLER_PATH = MODEL_ROOT / "latent_upscale_models" / "ltx-2.5-latent-spatial-upscaler-x2-bf16-1.0.safetensors"
DISTILLED_LORA_PATH = MODEL_ROOT / "loras" / "ltx-2.5-22b-distilled-lora-450-bf16.safetensors"

EFFECTIVE_STEPS = SMOKE_STEPS if SMOKE_TEST else TRAIN_STEPS

print(f"Experiment: {RUN_NAME}")
print(f"Repo ref:   {REPO_REF}")
print(f"Bucket:     {VIDEO_BUCKET} @ {FRAME_RATE:g} fps")
print(f"Steps:      {EFFECTIVE_STEPS} ({'smoke' if SMOKE_TEST else 'full'})")

## Runtime command helper

All external commands stream their output into the Colab cell and include an elapsed-time record. The helper passes only runtime settings to child processes; it never prints or persists a secret.

In [ ]:
def runtime_env(extra: dict[str, str] | None = None) -> dict[str, str]:
    env = os.environ.copy()
    env.update(
        {
            "HF_HOME": str(HF_CACHE_ROOT),
            "CUDA_VISIBLE_DEVICES": "0",
            "TOKENIZERS_PARALLELISM": "false",
            "PYTHONUNBUFFERED": "1",
        }
    )
    if extra:
        env.update(extra)
    return env


def run(
    cmd: list[str | Path],
    *,
    cwd: str | Path | None = None,
    env: dict[str, str] | None = None,
    display_cmd: list[str | Path] | None = None,
) -> subprocess.CompletedProcess[str]:
    cmd = [str(part) for part in cmd]
    shown = [str(part) for part in (display_cmd or cmd)]
    print(f"\n$ {shlex.join(shown)}")
    started = time.monotonic()
    completed = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=runtime_env(env),
        check=False,
    )
    elapsed = time.monotonic() - started
    print(f"[command exit={completed.returncode} elapsed={elapsed:.1f}s]")
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}: {shlex.join(shown)}")
    return completed


def check_file(path: Path, *, minimum_bytes: int = 1) -> None:
    if not path.is_file():
        raise FileNotFoundError(f"Required file is missing: {path}")
    if path.stat().st_size < minimum_bytes:
        raise RuntimeError(f"Required file is unexpectedly small: {path} ({path.stat().st_size} bytes)")


def count_pt_files(path: Path) -> int:
    return sum(1 for item in path.rglob("*.pt") if item.is_file())


RUN_ROOT.mkdir(parents=True, exist_ok=True)
HF_CACHE_ROOT.mkdir(parents=True, exist_ok=True)
print("Command helper ready.")
print(f"Run root: {RUN_ROOT}")

## 1. Colab preflight

Use a Linux Colab runtime with a CUDA GPU and enough local disk for the dev transformer, VAEs, text encoder, and precomputed cache. The trainer path uses bf16 and the repository's official natten extra. A failed preflight stops before downloading multi-gigabyte files.

In [ ]:
if not sys.platform.startswith("linux"):
    raise RuntimeError(f"This notebook expects Linux Colab; found {sys.platform!r}.")

if shutil.which("uv") is None:
    print("uv is not installed in this Colab runtime; installing it into the notebook environment.")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

if shutil.which("ffmpeg") is None or shutil.which("ffprobe") is None:
    raise RuntimeError("Colab must provide ffmpeg and ffprobe for media validation and first-frame extraction.")

run(["uv", "--version"])
if shutil.which("nvidia-smi"):
    run(["nvidia-smi"])
else:
    raise RuntimeError("nvidia-smi is unavailable; select a GPU runtime before continuing.")

WORK_ROOT.mkdir(parents=True, exist_ok=True)
disk = shutil.disk_usage(WORK_ROOT)
free_gib = disk.free / 2**30
print(f"Free disk at {WORK_ROOT}: {free_gib:.1f} GiB")
if free_gib < MIN_FREE_DISK_GIB:
    raise RuntimeError(
        f"Only {free_gib:.1f} GiB is free. At least {MIN_FREE_DISK_GIB} GiB is required "
        "for the split model pack and preprocessing cache."
    )

## 2. Clone and pin the reviewed LTX-2.5 source

The notebook uses a detached checkout of REPO_REF. It does not reset or delete an existing checkout. If a reused Colab directory is dirty, the notebook stops so an accidental local change cannot be hidden.

In [ ]:
REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    if not (REPO_DIR / ".git").is_dir():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git checkout; choose a new WORK_ROOT.")
    status = subprocess.check_output(
        ["git", "status", "--porcelain"],
        cwd=REPO_DIR,
        text=True,
    ).strip()
    if status:
        raise RuntimeError(
            f"{REPO_DIR} has uncommitted changes. Resolve them or choose a new WORK_ROOT; "
            "the notebook will not overwrite them."
        )
    run(["git", "fetch", "--no-tags", "origin", REPO_REF], cwd=REPO_DIR)
else:
    run(
        ["git", "clone", "--filter=blob:none", REPO_URL, str(REPO_DIR)],
        cwd=REPO_DIR.parent,
    )
    run(["git", "fetch", "--no-tags", "origin", REPO_REF], cwd=REPO_DIR)

run(["git", "checkout", "--detach", REPO_REF], cwd=REPO_DIR)
actual_ref = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO_DIR, text=True).strip()
if actual_ref != REPO_REF:
    raise RuntimeError(f"Checked out {actual_ref}, expected {REPO_REF}.")
run(["git", "status", "--short", "--branch"], cwd=REPO_DIR)

for required_path in (SOURCE_MANIFEST, CONFIG_TEMPLATE, TRAINER_DIR / "scripts" / "train.py"):
    check_file(required_path)
print("Pinned source and dataset are present.")

## 3. Install the repository environment

This is the repository's documented installation path. The notebook invokes the trainer through uv run, so the training process uses the resolved project environment rather than whichever packages happen to be in the Colab kernel.

In [ ]:
run(["uv", "sync", "--extra", "natten"], cwd=REPO_DIR)

torch_probe = """
import torch
import ltx_trainer
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available inside the uv environment.")
print("cuda runtime:", torch.version.cuda)
print("device:", torch.cuda.get_device_name(0))
print("device capability:", torch.cuda.get_device_capability(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())
if not torch.cuda.is_bf16_supported():
    raise RuntimeError("The selected GPU does not support bf16, required by this profile.")
props = torch.cuda.get_device_properties(0)
print("GPU VRAM GiB:", round(props.total_memory / 2**30, 1))
print("ltx_trainer import: OK")
"""
run(["uv", "run", "python", "-c", torch_probe], cwd=REPO_DIR)

## 4. Authenticate to Hugging Face

The LTX-2.5 files are gated. Accept the model terms in the browser first, then add a Colab secret named `HF_TOKEN` and enable notebook access for it. An existing `HF_TOKEN` environment variable takes precedence.

The token is copied into the runtime environment for Hugging Face download processes. It is not printed or written into this notebook or repository.

In [ ]:
hf_token = os.environ.get("HF_TOKEN")
if not hf_token:
    try:
        from google.colab import userdata
    except ImportError as exc:
        raise RuntimeError("HF_TOKEN is not set and Colab Secrets are unavailable.") from exc
    hf_token = userdata.get("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Add a Colab secret named HF_TOKEN and enable notebook access for it.")

os.environ["HF_TOKEN"] = hf_token
del hf_token
print("Hugging Face authentication is available from HF_TOKEN.")

## 5. Download the exact split pack required by the trainer

Training requires the dev transformer plus the matching Gemma 4 text encoder, video VAE, and audio VAE. The distilled transformer, official distilled LoRA, and spatial upsampler are downloaded only if the optional native pipeline smoke test is enabled later.

In [ ]:
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

required_model_files = [
    TRANSFORMER_PATH.relative_to(MODEL_ROOT).as_posix(),
    TEXT_ENCODER_PATH.relative_to(MODEL_ROOT).as_posix(),
    VIDEO_VAE_PATH.relative_to(MODEL_ROOT).as_posix(),
    AUDIO_VAE_PATH.relative_to(MODEL_ROOT).as_posix(),
]
print("Required model files:")
for relative_name in required_model_files:
    print("  ", relative_name)

run(
    [
        "uv",
        "run",
        "hf",
        "download",
        "Lightricks/LTX-2.5",
        *required_model_files,
        "--local-dir",
        str(MODEL_ROOT),
    ],
    cwd=REPO_DIR,
)

for model_path in (TRANSFORMER_PATH, TEXT_ENCODER_PATH, VIDEO_VAE_PATH, AUDIO_VAE_PATH):
    check_file(model_path, minimum_bytes=1024 * 1024)
    print(f"{model_path.name}: {model_path.stat().st_size / 2**30:.2f} GiB")

print("Required LTX-2.5 split pack is ready.")

## 6. Audit the raw manifest and make a deterministic holdout

The raw files in the repository are not changed. One clip is held out by a stable seed, and only the remaining clips appear in train_manifest.jsonl. The held-out clip supplies the validation image/audio and is never precomputed into the training root.

The processor extracts audio from each video entry itself. This is important: the video and speech remain one paired source, so their temporal origin cannot drift during dataset assembly.

In [ ]:
def read_jsonl(path: Path) -> list[dict]:
    rows = []
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSONL at {path}:{line_number}") from exc
            if not isinstance(row, dict):
                raise ValueError(f"Manifest row {line_number} is not an object.")
            rows.append(row)
    return rows


entries = read_jsonl(SOURCE_MANIFEST)
if not entries:
    raise RuntimeError(f"No entries found in {SOURCE_MANIFEST}.")

required_columns = {"video", "caption"}
for index, entry in enumerate(entries):
    missing = required_columns - entry.keys()
    if missing:
        raise ValueError(f"Manifest row {index} is missing columns: {sorted(missing)}")
    if not str(entry["caption"]).strip():
        raise ValueError(f"Manifest row {index} has an empty caption.")
    source_path = SOURCE_DATASET_DIR / str(entry["video"])
    check_file(source_path, minimum_bytes=1024)
    entry["_source_path"] = str(source_path.resolve())

video_names = [str(entry["video"]) for entry in entries]
if len(video_names) != len(set(video_names)):
    raise ValueError("Manifest contains duplicate video paths.")

if HOLDOUT_VIDEO_NAME is not None:
    matches = [entry for entry in entries if Path(str(entry["video"])).name == HOLDOUT_VIDEO_NAME]
    if len(matches) != 1:
        raise ValueError(f"HOLDOUT_VIDEO_NAME={HOLDOUT_VIDEO_NAME!r} did not identify exactly one entry.")
    holdout_entries = matches
else:
    if not 0 < HOLDOUT_COUNT < len(entries):
        raise ValueError(f"HOLDOUT_COUNT must be between 1 and {len(entries) - 1}.")
    holdout_entries = random.Random(HOLDOUT_SEED).sample(entries, HOLDOUT_COUNT)

holdout_keys = {str(entry["video"]) for entry in holdout_entries}
train_entries = [entry for entry in entries if str(entry["video"]) not in holdout_keys]
if not train_entries:
    raise RuntimeError("The training split is empty.")

# Keep only the public manifest schema in generated manifests.
def runtime_row(entry: dict) -> dict:
    return {
        "video": f"media/{Path(str(entry['video'])).name}",
        "caption": str(entry["caption"]),
    }


RUNTIME_MEDIA_DIR.mkdir(parents=True, exist_ok=True)
all_split_entries = train_entries + holdout_entries
destination_names = [Path(str(entry["video"])).name for entry in all_split_entries]
if len(destination_names) != len(set(destination_names)):
    raise ValueError("Basenames collide when staging the split; rename the source files first.")

for entry in all_split_entries:
    source_path = Path(entry["_source_path"])
    destination = RUNTIME_MEDIA_DIR / source_path.name
    if destination.is_symlink() or destination.exists():
        if destination.is_dir():
            raise RuntimeError(f"Refusing to replace directory: {destination}")
        destination.unlink()
    destination.symlink_to(source_path)

TRAIN_MANIFEST.write_text(
    "\n".join(json.dumps(runtime_row(entry), ensure_ascii=False) for entry in train_entries) + "\n",
    encoding="utf-8",
)
HOLDOUT_MANIFEST.write_text(
    "\n".join(json.dumps(runtime_row(entry), ensure_ascii=False) for entry in holdout_entries) + "\n",
    encoding="utf-8",
)

split_summary = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "source_manifest": str(SOURCE_MANIFEST),
    "seed": HOLDOUT_SEED,
    "train_count": len(train_entries),
    "holdout_count": len(holdout_entries),
    "train_videos": [str(entry["video"]) for entry in train_entries],
    "holdout_videos": [str(entry["video"]) for entry in holdout_entries],
}
SPLIT_SUMMARY.write_text(json.dumps(split_summary, indent=2) + "\n", encoding="utf-8")

print(f"Raw entries: {len(entries)}")
print(f"Training entries: {len(train_entries)}")
print(f"Holdout entries: {len(holdout_entries)}")
print("Holdout:")
for entry in holdout_entries:
    print("  ", entry["video"])

### Media stream audit

Every training source must have one usable video stream and one audio stream. This catches silent clips, corrupt downloads, and accidental multi-stream files before expensive VAE encoding.

In [ ]:
def ffprobe_json(path: Path) -> dict:
    completed = subprocess.run(
        [
            "ffprobe",
            "-v",
            "error",
            "-show_entries",
            "format=duration:stream=index,codec_type,width,height,r_frame_rate,duration",
            "-of",
            "json",
            str(path),
        ],
        check=False,
        capture_output=True,
        text=True,
    )
    if completed.returncode != 0:
        raise RuntimeError(f"ffprobe failed for {path}: {completed.stderr.strip()}")
    return json.loads(completed.stdout)


for entry in entries:
    path = Path(entry["_source_path"])
    info = ffprobe_json(path)
    streams = info.get("streams", [])
    video_streams = [stream for stream in streams if stream.get("codec_type") == "video"]
    audio_streams = [stream for stream in streams if stream.get("codec_type") == "audio"]
    if len(video_streams) != 1:
        raise ValueError(f"{path.name}: expected one video stream, found {len(video_streams)}")
    if len(audio_streams) != 1:
        raise ValueError(f"{path.name}: expected one audio stream, found {len(audio_streams)}")
    width = video_streams[0].get("width")
    height = video_streams[0].get("height")
    duration = float(info.get("format", {}).get("duration") or 0.0)
    print(f"{path.name}: {width}x{height}, {duration:.3f}s, audio=present")

print("Media stream audit passed.")

## 7. First-frame QA for the held-out example

The training condition is the first decoded frame of the source video. Review frame 0 plus frames around 0.2s, 0.4s, and 1.0s, together with the audio, before preprocessing:

- one intended speaker is visible;
- face, eyes, lips, and chin are sharp and unobstructed;
- there is no cut, fade, subtitle, watermark, blur, or speaker entrance at frame 0;
- the mouth begins in a natural pose;
- framing and exposure stay stable;
- speech is not clipped at the beginning and remains aligned to the visible speech.

If this held-out example fails, select a better holdout or fix the source dataset before training. Do not generate an artificial first frame to hide a bad source frame.

In [ ]:
holdout_source = Path(holdout_entries[0]["_source_path"])
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

run(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(holdout_source),
        "-vf",
        r"select=eq(n\,0)",
        "-frames:v",
        "1",
        str(VAL_IMAGE),
    ]
)
run(
    [
        "ffmpeg",
        "-hide_banner",
        "-loglevel",
        "error",
        "-y",
        "-i",
        str(holdout_source),
        "-map",
        "0:a:0",
        "-vn",
        "-ac",
        "2",
        "-ar",
        "48000",
        "-c:a",
        "pcm_s16le",
        str(VAL_AUDIO),
    ]
)
check_file(VAL_IMAGE, minimum_bytes=1024)
check_file(VAL_AUDIO, minimum_bytes=1024)
audio_probe = json.loads(
    subprocess.check_output(
        [
            "ffprobe",
            "-v",
            "error",
            "-select_streams",
            "a:0",
            "-show_entries",
            "stream=channels,sample_rate",
            "-of",
            "json",
            str(VAL_AUDIO),
        ],
        text=True,
    )
)
audio_stream = audio_probe["streams"][0]
if int(audio_stream["channels"]) != 2 or int(audio_stream["sample_rate"]) != 48000:
    raise RuntimeError(f"Validation audio must be stereo 48 kHz for the LTX audio VAE: {audio_stream}")
print("Validation audio audit: stereo, 48000 Hz")

qa_frames = [(0.0, VAL_IMAGE)]
for timestamp in (0.2, 0.4, 1.0):
    qa_path = VALIDATION_DIR / f"qa_{timestamp:.1f}s.png"
    run(
        [
            "ffmpeg",
            "-hide_banner",
            "-loglevel",
            "error",
            "-y",
            "-i",
            str(holdout_source),
            "-ss",
            f"{timestamp:.2f}",
            "-frames:v",
            "1",
            str(qa_path),
        ]
    )
    check_file(qa_path, minimum_bytes=1024)
    qa_frames.append((timestamp, qa_path))

from IPython.display import Audio, Image, display
for timestamp, frame_path in qa_frames:
    print(f"QA frame at {timestamp:.1f}s")
    display(Image(filename=str(frame_path)))
display(Audio(filename=str(VAL_AUDIO)))
print(f"Validation source: {holdout_source.name}")
print(f"Validation image:  {VAL_IMAGE}")
print(f"Validation audio:  {VAL_AUDIO}")

## 8. Precompute fresh LTX-2.5 training features

The official processor creates:

- latents/: target video VAE latents;
- audio_latents/: audio automatically extracted from the same video and encoded with the matching audio VAE;
- conditions/: Gemma 4 text embeddings.

This profile intentionally does not pass the deprecated with-audio flag; audio extraction is enabled by default when the manifest has only video and caption.

A run-specific preprocessing specification prevents accidentally reusing a cache made with a different model or bucket. Existing caches are never silently overwritten.

In [ ]:
PRECOMPUTED_DIR.mkdir(parents=True, exist_ok=True)

def file_signature(path: Path) -> dict:
    stat = path.stat()
    return {"path": str(path), "size": stat.st_size, "mtime_ns": stat.st_mtime_ns}


preprocess_spec = {
    "repo_ref": actual_ref,
    "train_manifest": str(TRAIN_MANIFEST),
    "train_manifest_sha256": hashlib.sha256(TRAIN_MANIFEST.read_bytes()).hexdigest(),
    "resolution_bucket": VIDEO_BUCKET,
    "frame_rate": FRAME_RATE,
    "model_files": {
        "transformer": file_signature(TRANSFORMER_PATH),
        "text_encoder": file_signature(TEXT_ENCODER_PATH),
        "video_vae": file_signature(VIDEO_VAE_PATH),
        "audio_vae": file_signature(AUDIO_VAE_PATH),
    },
}

existing_pt_files = count_pt_files(PRECOMPUTED_DIR)
if PREPROCESS_SPEC.exists():
    saved_spec = json.loads(PREPROCESS_SPEC.read_text(encoding="utf-8"))
    if saved_spec != preprocess_spec and existing_pt_files and not PRECOMPUTE_OVERWRITE:
        raise RuntimeError(
            f"Existing precompute cache at {PRECOMPUTED_DIR} does not match this run. "
            "Use a new RUN_NAME or set PRECOMPUTE_OVERWRITE=True after reviewing the change."
        )
elif existing_pt_files and not PRECOMPUTE_OVERWRITE:
    raise RuntimeError(
        f"Existing precompute files at {PRECOMPUTED_DIR} have no specification. "
        "Use a new RUN_NAME or set PRECOMPUTE_OVERWRITE=True."
    )

PREPROCESS_SPEC.write_text(json.dumps(preprocess_spec, indent=2) + "\n", encoding="utf-8")

preprocess_cmd = [
    "uv",
    "run",
    "python",
    "scripts/process_dataset.py",
    str(TRAIN_MANIFEST),
    "--resolution-buckets",
    VIDEO_BUCKET,
    "--model-path",
    str(TRANSFORMER_PATH),
    "--text-encoder-path",
    str(TEXT_ENCODER_PATH),
    "--video-vae-path",
    str(VIDEO_VAE_PATH),
    "--audio-vae-path",
    str(AUDIO_VAE_PATH),
    "--video-column",
    "video",
    "--caption-column",
    "caption",
    "--output-dir",
    str(PRECOMPUTED_DIR),
    "--device",
    "cuda",
    "--batch-size",
    str(PREPROCESS_BATCH_SIZE),
]
if PRECOMPUTE_OVERWRITE:
    preprocess_cmd.append("--overwrite")

run(preprocess_cmd, cwd=TRAINER_DIR)

expected_count = len(train_entries)
for role in ("latents", "audio_latents", "conditions"):
    role_dir = PRECOMPUTED_DIR / role
    count = count_pt_files(role_dir)
    print(f"{role}: {count} .pt files")
    if count != expected_count:
        raise RuntimeError(f"{role} has {count} items but the training split has {expected_count}.")

print("Precomputed feature audit passed.")

### Precomputed tensor sanity check

This checks representative keys and shapes inside the actual uv environment. It does not load the 22B transformer; it only verifies that the cache is readable and structurally complete.

In [ ]:
sample_latent = sorted((PRECOMPUTED_DIR / "latents").rglob("*.pt"))[0]
sample_audio = sorted((PRECOMPUTED_DIR / "audio_latents").rglob("*.pt"))[0]
sample_condition = sorted((PRECOMPUTED_DIR / "conditions").rglob("*.pt"))[0]

inspect_code = f"""
from pathlib import Path
import torch

paths = {{
    "latents": Path({str(sample_latent)!r}),
    "audio_latents": Path({str(sample_audio)!r}),
    "conditions": Path({str(sample_condition)!r}),
}}
for name, path in paths.items():
    value = torch.load(path, map_location="cpu", weights_only=True)
    if isinstance(value, dict):
        summary = {{key: (tuple(item.shape) if hasattr(item, "shape") else type(item).__name__) for key, item in value.items()}}
        print(name, path.name, summary)
    else:
        print(name, path.name, type(value).__name__, getattr(value, "shape", None))
"""
run(["uv", "run", "python", "-c", inspect_code], cwd=REPO_DIR)

## 9. Materialize and validate the run configuration

The source YAML remains untouched. This cell fills only the paths and experiment controls for this Colab run, keeps the trainer's explicit A2V conditioning, and writes an auditable copy under the run directory.

The configuration retains:

- clean paired audio as the non-generated modality;
- video as the generated modality;
- first-frame conditioning probability 1.0;
- explicit LoRA targets for video self-attention, video/text attention, and audio-to-video attention;
- bf16, no quantization, gradient checkpointing, and one-sample batches;
- diagnostics that print batch alignment and gradient statistics.

In [ ]:
try:
    import yaml
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml"], check=True)
    import yaml

config = yaml.safe_load(CONFIG_TEMPLATE.read_text(encoding="utf-8"))
if not isinstance(config, dict):
    raise ValueError("Trainer template did not parse as a mapping.")
video_training = config["training_strategy"]["video"]
audio_training = config["training_strategy"]["audio"]
if not video_training["is_generated"] or audio_training["is_generated"]:
    raise ValueError("The selected profile must generate video and keep audio clean/frozen.")
first_frame_conditions = [item for item in video_training["conditions"] if item["type"] == "first_frame"]
if len(first_frame_conditions) != 1 or first_frame_conditions[0].get("probability") != 1.0:
    raise ValueError("The selected profile must apply first-frame conditioning with probability 1.0.")
if not any(item.startswith("audio_to_video_attn.") for item in config["lora"]["target_modules"]):
    raise ValueError("The selected LoRA profile has no audio-to-video attention targets.")

config["model"]["model_path"] = str(TRANSFORMER_PATH)
config["model"]["text_encoder_path"] = str(TEXT_ENCODER_PATH)
config["model"]["video_vae_path"] = str(VIDEO_VAE_PATH)
config["model"]["audio_vae_path"] = str(AUDIO_VAE_PATH)
config["lora"]["rank"] = LORA_RANK
config["lora"]["alpha"] = LORA_ALPHA
config["optimization"]["learning_rate"] = LEARNING_RATE
existing_checkpoints = sorted((OUTPUT_DIR / "checkpoints").glob("*.safetensors"))
if START_FRESH:
    config["model"]["load_checkpoint"] = None
elif existing_checkpoints:
    config["model"]["load_checkpoint"] = str(existing_checkpoints[-1])
else:
    raise RuntimeError(
        "START_FRESH=False but no checkpoint exists to resume. Run a smoke test first or use a new run."
    )

config["optimization"]["steps"] = EFFECTIVE_STEPS
config["optimization"]["batch_size"] = 1
config["optimization"]["enable_gradient_checkpointing"] = True

config["data"]["preprocessed_data_root"] = str(PRECOMPUTED_DIR)
config["data"]["num_dataloader_workers"] = NUM_DATALOADER_WORKERS

config["validation"]["video_dims"] = [WIDTH, HEIGHT, NUM_FRAMES]
config["validation"]["frame_rate"] = FRAME_RATE
config["validation"]["interval"] = (10 if SMOKE_TEST else 100) if RUN_TRAINER_VALIDATION else None
config["validation"]["samples"][0]["conditions"] = [
    {"type": "first_frame", "image_or_video": str(VAL_IMAGE)},
    {"type": "audio_to_video", "audio": str(VAL_AUDIO)},
]
config["validation"]["generate_audio"] = False
config["validation"]["generate_video"] = True
config["validation"]["skip_initial_validation"] = not RUN_TRAINER_VALIDATION

config["checkpoints"]["interval"] = 10 if SMOKE_TEST else 100
config["checkpoints"]["keep_last_n"] = -1
config["checkpoints"]["precision"] = "bfloat16"
config["checkpoints"]["no_resume"] = START_FRESH
config["checkpoints"]["save_training_state"] = "minimal"

config["wandb"]["enabled"] = False
config["seed"] = 42
config["output_dir"] = str(OUTPUT_DIR)

serialized = yaml.safe_dump(config, sort_keys=False)
if "path/to/" in serialized:
    raise RuntimeError("A placeholder path remains in the generated trainer config.")
RUN_CONFIG.parent.mkdir(parents=True, exist_ok=True)
RUN_CONFIG.write_text(serialized, encoding="utf-8")

run_metadata = {
    "created_at_utc": dt.datetime.now(dt.timezone.utc).isoformat(),
    "repo_url": REPO_URL,
    "repo_ref": actual_ref,
    "run_name": RUN_NAME,
    "smoke_test": SMOKE_TEST,
    "effective_steps": EFFECTIVE_STEPS,
    "bucket": VIDEO_BUCKET,
    "frame_rate": FRAME_RATE,
    "train_manifest": str(TRAIN_MANIFEST),
    "holdout_manifest": str(HOLDOUT_MANIFEST),
    "precomputed_dir": str(PRECOMPUTED_DIR),
    "validation_image": str(VAL_IMAGE),
    "validation_audio": str(VAL_AUDIO),
    "model_files": {
        "transformer": str(TRANSFORMER_PATH),
        "text_encoder": str(TEXT_ENCODER_PATH),
        "video_vae": str(VIDEO_VAE_PATH),
        "audio_vae": str(AUDIO_VAE_PATH),
    },
}
RUN_METADATA.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")

if START_FRESH and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise RuntimeError(
        f"{OUTPUT_DIR} is not empty. Choose a new RUN_NAME for a fresh run, "
        "or set START_FRESH=False to resume deliberately."
    )

validate_code = f"""
from pathlib import Path
import yaml
from ltx_trainer.config import LtxTrainerConfig

path = Path({str(RUN_CONFIG)!r})
parsed = yaml.safe_load(path.read_text())
validated = LtxTrainerConfig(**parsed)
print("validated output_dir:", validated.output_dir)
print("validated training mode:", validated.model.training_mode)
print("validated video conditions:", [item.type for item in validated.training_strategy.video.conditions])
print("validated audio generated:", validated.training_strategy.audio.is_generated)
"""
run(["uv", "run", "python", "-c", validate_code], cwd=TRAINER_DIR)

print(f"Run config: {RUN_CONFIG}")
print(f"Run metadata: {RUN_METADATA}")

## 10. Run the native single-GPU trainer

Run the smoke test first. This calls scripts/train.py directly, as required by the trainer's single-GPU path; it does not launch accelerate or a multi-GPU wrapper.

For the full experiment after a successful smoke run, keep RUN_NAME unchanged, set SMOKE_TEST=False and START_FRESH=False, rerun the configuration cell, and then run this cell; the latest smoke checkpoint and minimal training state will be used for resume. For an independent full run, choose a new RUN_NAME with START_FRESH=True. If any run is interrupted, keep its run directory, set START_FRESH=False, and rerun the configuration and training cells.

In [ ]:
if START_FRESH and OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()):
    raise RuntimeError("Fresh run requested but the output directory is not empty; choose a new RUN_NAME.")

train_started = dt.datetime.now(dt.timezone.utc).isoformat()
print(f"Training started (UTC): {train_started}")
run(
    ["uv", "run", "python", "scripts/train.py", str(RUN_CONFIG)],
    cwd=TRAINER_DIR,
)
print(f"Training completed (UTC): {dt.datetime.now(dt.timezone.utc).isoformat()}")

## 11. Inspect the produced LoRA checkpoint

The trainer writes a ComfyUI-compatible .safetensors LoRA under outputs/checkpoints. The inspection below checks that it is non-empty, readable, contains tensors, and uses the expected diffusion_model. key namespace emitted by this fork.

In [ ]:
checkpoints = sorted((OUTPUT_DIR / "checkpoints").glob("*.safetensors"))
if not checkpoints:
    raise RuntimeError(f"No LoRA checkpoint found under {OUTPUT_DIR / 'checkpoints'}.")

latest_lora = checkpoints[-1]
print("Available checkpoints:")
for path in checkpoints:
    print(f"  {path.name}: {path.stat().st_size / 2**20:.1f} MiB")

inspect_lora_code = f"""
from safetensors import safe_open

path = {str(latest_lora)!r}
with safe_open(path, framework="pt", device="cpu") as handle:
    keys = list(handle.keys())
    print("tensor count:", len(keys))
    print("first keys:", keys[:8])
    if not keys:
        raise RuntimeError("LoRA checkpoint has no tensors.")
    unexpected = [key for key in keys if not key.startswith("diffusion_model.")]
    if unexpected:
        raise RuntimeError(f"Unexpected non-ComfyUI LoRA keys: {{unexpected[:5]}}")
    print("key namespace: diffusion_model.*")
"""
run(["uv", "run", "python", "-c", inspect_lora_code], cwd=REPO_DIR)
print(f"Latest LoRA: {latest_lora}")

## 12. Optional native LTX-2.5 A2V pipeline smoke test

This optional cell validates the trained LoRA through the repository's native two-stage A2V pipeline. It additionally downloads the official distilled LoRA and spatial upsampler. The trained checkpoint is passed as a custom LoRA; the official distilled LoRA is a separate inference component and is not the trained result.

This is intentionally opt-in because it loads the full dev pipeline and performs generation. Set RUN_PIPELINE_SMOKE_TEST=True before running the cell.

In [ ]:
if not RUN_PIPELINE_SMOKE_TEST:
    print("Native pipeline smoke test is disabled. Set RUN_PIPELINE_SMOKE_TEST=True and rerun this cell to enable it.")
else:
    optional_model_files = [
        DISTILLED_LORA_PATH.relative_to(MODEL_ROOT).as_posix(),
        SPATIAL_UPSAMPLER_PATH.relative_to(MODEL_ROOT).as_posix(),
    ]
    run(
        [
            "uv",
            "run",
            "hf",
            "download",
            "Lightricks/LTX-2.5",
            *optional_model_files,
            "--local-dir",
            str(MODEL_ROOT),
        ],
        cwd=REPO_DIR,
    )
    check_file(DISTILLED_LORA_PATH, minimum_bytes=1024 * 1024)
    check_file(SPATIAL_UPSAMPLER_PATH, minimum_bytes=1024 * 1024)

    pipeline_cmd = [
        "uv",
        "run",
        "python",
        "-m",
        "ltx_pipelines.a2vid_two_stage",
        "--transformer-path",
        str(TRANSFORMER_PATH),
        "--text-encoder-path",
        str(TEXT_ENCODER_PATH),
        "--video-vae-path",
        str(VIDEO_VAE_PATH),
        "--audio-vae-path",
        str(AUDIO_VAE_PATH),
        "--spatial-upsampler-path",
        str(SPATIAL_UPSAMPLER_PATH),
        "--distilled-lora",
        str(DISTILLED_LORA_PATH),
        "1.0",
        "--lora",
        str(latest_lora),
        "1.0",
        "--audio-path",
        str(VAL_AUDIO),
        "--image",
        str(VAL_IMAGE),
        "0",
        "0.7",
        "--prompt",
        "A close-up talking-head video of a person looking directly into the camera and speaking naturally with clear mouth motion and controlled facial expression.",
        "--negative-prompt",
        "blurry, flickering, deformed face, distorted mouth, mismatched lip sync, camera shake, subtitles, watermark",
        "--num-frames",
        str(NUM_FRAMES),
        "--frame-rate",
        str(FRAME_RATE),
        "--width",
        str(WIDTH),
        "--height",
        str(HEIGHT),
        "--num-inference-steps",
        "30",
        "--seed",
        "42",
        "--output-path",
        str(PIPELINE_OUTPUT),
    ]
    run(pipeline_cmd, cwd=REPO_DIR)
    check_file(PIPELINE_OUTPUT, minimum_bytes=1024)
    from IPython.display import Video, display
    display(Video(str(PIPELINE_OUTPUT), embed=False))
    print(f"Native A2V output: {PIPELINE_OUTPUT}")

## 13. Export and handoff

For ComfyUI, copy the trained .safetensors into ComfyUI/models/loras and load it in the official LTX-2.5 A2V workflow. The checkpoint produced here is the custom training result; do not confuse it with Lightricks' official distilled LoRA.

Keep the following artifacts together when evaluating a run:

- run_config.yaml;
- run_metadata.json;
- dataset/train_manifest.jsonl;
- dataset/holdout_manifest.jsonl;
- dataset/split_summary.json;
- precomputed/preprocess_spec.json;
- outputs/checkpoints/*.safetensors;
- validation image/audio and any validation samples.

The optional download cell uses Colab's browser transfer rather than uploading model or dataset files to another service.

In [ ]:
print(f"LoRA checkpoint for export: {latest_lora}")
print("Download is disabled by default.")
if DOWNLOAD_LORA:
    try:
        from google.colab import files
        files.download(str(latest_lora))
    except ImportError as exc:
        raise RuntimeError("DOWNLOAD_LORA=True but this is not a Google Colab runtime.") from exc
else:
    print("Set DOWNLOAD_LORA=True and rerun this cell to download the checkpoint.")

## Handoff: artifacts, debugging, and acceptance

### Artifact ownership

The repository owns the reviewed source code, the raw training clips, and the source dataset manifest. Colab owns the run directory, model cache, staged manifests, precomputed tensors, validation media, logs printed by the trainer, and generated checkpoints. Save the complete run directory before disconnecting the runtime. The LoRA file alone is not enough to reproduce or judge an experiment.

The most important files are run_config.yaml for the exact effective trainer configuration, run_metadata.json for the code/model/run provenance, split_summary.json for the train/holdout membership, preprocess_spec.json for the model and bucket used to create the cache, and the checkpoint plus validation samples. The staged media are symlinks to the raw repository clips; they are not a second copy of the dataset.

### Debugging order

If preprocessing fails, inspect the last media filename printed by the processor and rerun the stream audit on that source. If the cache audit reports unequal counts, do not start training; fix the failed item or use a new run name. If the cache specification differs, do not force reuse until the model revision, manifest hash, bucket, and VAE paths have been reviewed. PRECOMPUTE_OVERWRITE=True is an explicit repair operation, not a default recovery switch.

If configuration validation fails, treat it as a schema or path problem. Confirm that the four split model files exist, the three precomputed directories are populated, the validation image and audio exist, and the generated YAML contains no placeholder path. If training fails with an out-of-memory error, reduce the bucket consistently in VIDEO_BUCKET, WIDTH, and HEIGHT and recompute into a new run; changing only the YAML would make the cache and config inconsistent. Do not add quantization to the first high-VRAM experiment without recording that as a different experiment.

If resume is needed, use the same RUN_NAME and set START_FRESH=False. The notebook selects the latest LoRA checkpoint, checks the saved training state, and lets the trainer decide whether the optimizer and scheduler state is compatible. If no checkpoint is present, stop and use a new run or complete the smoke run first.

The optional native pipeline has its own expected failure class: it requires the official distilled LoRA and spatial upsampler in addition to the dev model pack. A missing optional file is not a training failure. The custom checkpoint trained here and the official distilled LoRA must remain separate arguments.

### Acceptance standard

Accept the implementation only after the processor counts match, the trainer prints the intended clean-audio/generated-video modality contract, the first-frame conditioning and A2V target audit pass, the checkpoint is readable, and the held-out sample has stable identity, natural mouth motion, and correct word timing. Compare checkpoints at multiple steps; choose by held-out behavior, not the lowest training loss. With 35 short clips and one holdout, this is a controlled first experiment. It is not sufficient evidence that arbitrary people, images, accents, or speech will achieve production lip synchronization.

### Decision gates

Do not interpret a lower training loss alone as success. The first experiment passes only if:

1. the precompute audit has equal counts for video, audio, and text features;
2. the smoke checkpoint is readable and has the expected key namespace;
3. held-out validation produces a stable face with audio that starts at the intended word;
4. the mouth motion follows held-out speech rather than merely copying a training identity;
5. the same checkpoint remains usable at the intended inference resolution and duration.

With this dataset size, treat the result as an objective/implementation smoke test, not evidence of broad generalization. A convincing held-out result requires more speakers, varied phonetic content, and a validation split that is not just a near-duplicate of a training speaker.